# Plasmidsaurus 3′ end RNA-seq preprocessing and quantification

## Overview

This Jupyter notebook implements the preprocessing and gene expression quantification workflow for Plasmidsaurus 3′ end RNA sequencing data.  
The analysis was applied to investigate transcriptional differences in *Ménage à trois* mating experiments, as described in our preprint:

https://www.biorxiv.org/content/10.1101/2025.02.11.637763v2

The workflow implemented here was informed by and adapted from the publicly available Plasmidsaurus RNA-seq analysis documentation:
https://plasmidsaurus.com/technical-documentation/rna

This notebook covers read preprocessing, alignment, UMI-based deduplication, mapping quality control, and gene expression quantification.  
Downstream analyses such as principal component analysis (PCA) and differential gene expression (DEG) analysis are performed separately and are not included here.


## Step 1 Unzip raw sequencing files

Raw sequencing files are provided as compressed archives and are decompressed prior to downstream processing.

```bash
unzip /path/to/your/project/Rawdata/P55ST4_fastq.zip
```

## Step 2 Read preprocessing with FastP using SLURM array jobs

This step performs read filtering and trimming using FastP in a high performance computing environment.  
A SLURM array job is used to process multiple samples in parallel.

### Overview
- Each array task corresponds to one sequencing sample
- Raw FASTQ files are processed individually
- Filtered reads and quality control reports are generated for each sample

### Key parameters
- Poly X tail trimming and low quality tail trimming are enabled
- Reads shorter than 50 bp after trimming are discarded
- Bases with Phred quality scores below 15 are filtered
- HTML and JSON QC reports are generated for each sample

### Input and output
- **Input** Raw FASTQ files located in the raw data directory
- **Output** Filtered FASTQ files and FastP QC reports


In [ ]:
%%writefile /path/to/your/project/trimming/20260125_trim_step.sb
#!/bin/bash
#SBATCH --array=0-5
#SBATCH --cpus-per-task=8
#SBATCH --time=23:59:00
#SBATCH --mem-per-cpu=1500mb
#SBATCH --output=/path/to/your/project/trimming/20260125_trim_step.%a.out


QRYlist=(P55ST4_1_WT-1 P55ST4_2_WT-2 P55ST4_3_WT-3 P55ST4_4_S-1 P55ST4_5_S-2 P55ST4_6_S-3)
QRY="${QRYlist[$SLURM_ARRAY_TASK_ID]}"


RAW_DIR=/path/to/your/project/Rawdate
OUT_DIR=/path/to/your/project/trimming/filtered_reads
QC_DIR=/path/to/your/project/trimming/fastp_qc


mkdir -p "$OUT_DIR" "$QC_DIR"


cd "$RAW_DIR" || exit 1


fastp \
-i "./${QRY}.fastq.gz" \
-o "${OUT_DIR}/${QRY}.fastp.filtered.fastq.gz" \
--trim_poly_x \
--cut_tail \
--cut_tail_mean_quality 15 \
--length_required 50 \
--qualified_quality_phred 15 \
--thread "${SLURM_CPUS_PER_TASK}" \
--html "${QC_DIR}/${QRY}.fastp.html" \
--json "${QC_DIR}/${QRY}.fastp.json"

## Step 2.5 Determine read length distribution after trimming

After read preprocessing, the read length distribution is examined to determine the appropriate `sjdbOverhang` parameter for STAR genome index generation.

### Overview
- Read length distributions are calculated from trimmed FASTQ files
- The dominant read length is used to guide STAR index configuration

### Example directory

### What this step does
For each `*.fastp.filtered.fastq.gz` file, the script reports the five most frequent read lengths observed in the dataset.

In this dataset, the dominant read length is **94 bp** across all six samples, which informs the choice of `sjdbOverhang = 93`.

### Command used
```bash
for f in /path/to/your/project/trimming/filtered_reads/*.fastp.filtered.fastq.gz
do
  echo "==== $(basename "$f") ===="
  zcat "$f" \
    | awk 'NR%4==2{print length($0)}' \
    | sort -n \
    | uniq -c \
    | tail -n 5
done

## Step 3 STAR genome index generation

This step generates a STAR genome index for read alignment.  
A customized reference genome FASTA file and a corresponding GTF annotation file are used to ensure consistent gene identifiers during downstream quantification.

### Overview
- A STAR genome index is built once per reference genome
- The index incorporates splice junction annotations from the GTF file
- This step is required prior to read alignment

### Key parameters
- `sjdbOverhang` is set to read length minus one to optimize splice junction detection
- Multi thread execution is enabled using SLURM allocated CPUs

### Input and output
- **Input** Reference genome FASTA file and GTF annotation file  
- **Output** STAR genome index files stored in the specified index directory

> **Note**  
> Reference genome paths and index locations reflect a local computing environment and should be adapted as needed. The `sjdbOverhang` parameter should be adjusted according to the effective read length after trimming.

In [ ]:
%%writefile /path/to/your/project/mapping/20260125_star_index_fixed_step.sb
#!/bin/bash
#SBATCH --cpus-per-task=12
#SBATCH --time=23:59:00
#SBATCH --mem-per-cpu=4000mb
#SBATCH --output=/path/to/your/project/mapping/20260125_star_index_fixed_step.out

INDEX_DIR=/path/to/your/project/reference/STAR_index/XL280alpha_JEC20MATa_MATageneID_fixed
mkdir -p "$INDEX_DIR"

# TODO 1: change these two paths to your real reference files
GENOME_FA=/path/to/your/project/Genome_annotation/modified_sorted_XL280alpha_sequence_rename+MATa.fasta
GENOME_GTF=/path/to/your/project/Genome_annotation/XL280_liftmerged_geneID_fixed.gtf

# TODO 2: set sjdbOverhang = (read_length - 1)
# If your reads are ~94bp after trimming, keep 93.
SJDB_OVERHANG=93

if [[ ! -s "$GENOME_FA" ]]; then
  echo "ERROR. genome fasta not found: $GENOME_FA"
  exit 1
fi

if [[ ! -s "$GENOME_GTF" ]]; then
  echo "ERROR. genome gtf not found: $GENOME_GTF"
  exit 1
fi

STAR --version || true

STAR \
  --runMode genomeGenerate \
  --runThreadN "${SLURM_CPUS_PER_TASK}" \
  --genomeDir "${INDEX_DIR}" \
  --genomeFastaFiles "${GENOME_FA}" \
  --sjdbGTFfile "${GENOME_GTF}" \
  --sjdbOverhang "${SJDB_OVERHANG}"

echo "Done. STAR genome index created at: ${INDEX_DIR}"

## Step 4 Read alignment using STAR with SLURM array jobs

This step aligns trimmed reads to the reference genome using the STAR aligner in a high performance computing environment.  
A SLURM array job is used to process multiple samples in parallel.

### Overview
- Each SLURM array task corresponds to one sequencing sample
- Trimmed reads are aligned using a pre generated STAR genome index
- Unsorted BAM files are generated and subsequently sorted and indexed
- Unmapped reads are exported for optional downstream inspection

### Key parameters and options
- Multi thread execution is enabled using SLURM allocated CPUs
- Non canonical splice junctions are filtered during alignment
- Unmapped reads are output in FASTQ format

### Input and output
- **Input** Trimmed FASTQ files generated by FastP  
- **Output** Coordinate sorted and indexed BAM files, along with unmapped reads

> **Note**  
> File paths and SLURM resource specifications shown here reflect a local computing environment and should be adapted as needed. The STAR genome index used in this step must be generated in advance.

In [ ]:
%%writefile /path/to/your/project/mapping/20260125_star_map_step.sb
#!/bin/bash
#SBATCH --array=0-5
#SBATCH --cpus-per-task=12
#SBATCH --time=23:59:00
#SBATCH --mem-per-cpu=2000mb
#SBATCH --output=/path/to/your/project/mapping/20260125_star_map_step.%a.out

QRYlist=(P55ST4_1_WT-1 P55ST4_2_WT-2 P55ST4_3_WT-3 P55ST4_4_S-1 P55ST4_5_S-2 P55ST4_6_S-3)
TASK_ID=${SLURM_ARRAY_TASK_ID:-}
if [[ -z "${TASK_ID}" ]]; then
  echo "ERROR. SLURM_ARRAY_TASK_ID is not set. This job is not running as an array."
  exit 1
fi
QRY="${QRYlist[$TASK_ID]}"

TRIM_DIR=/path/to/your/project/trimming/filtered_reads
MAP_DIR=/path/to/your/project/mapping/star
UNMAP_DIR=/path/to/your/project/mapping/unmapped_fastx

mkdir -p "$MAP_DIR" "$UNMAP_DIR"

IN_FASTQ="${TRIM_DIR}/${QRY}.fastp.filtered.fastq.gz"
if [[ ! -s "$IN_FASTQ" ]]; then
  echo "ERROR. Input fastq not found or empty. $IN_FASTQ"
  exit 1
fi

cd "$MAP_DIR" || exit 1

STAR_INDEX_DIR=/path/to/your/project/reference/STAR_index/XL280alpha_JEC20MATa_MATageneID_fixed

STAR \
  --runThreadN "${SLURM_CPUS_PER_TASK}" \
  --genomeDir "${STAR_INDEX_DIR}" \
  --readFilesIn "${IN_FASTQ}" \
  --readFilesCommand zcat \
  --outFileNamePrefix "${QRY}." \
  --outSAMtype BAM Unsorted \
  --outFilterIntronMotifs RemoveNoncanonical \
  --outReadsUnmapped Fastx

if [[ ! -s "${QRY}.Aligned.out.bam" ]]; then
  echo "ERROR. STAR did not produce BAM for ${QRY}"
  exit 1
fi

samtools sort \
  -@ "${SLURM_CPUS_PER_TASK}" \
  -o "${QRY}.Aligned.sortedByCoord.out.bam" \
  "${QRY}.Aligned.out.bam"

samtools index -@ "${SLURM_CPUS_PER_TASK}" "${QRY}.Aligned.sortedByCoord.out.bam"

if [[ -s "${QRY}.Unmapped.out.mate1" ]]; then
  mv "${QRY}.Unmapped.out.mate1" "${UNMAP_DIR}/${QRY}.Unmapped.fastq"
fi

echo "Done. ${QRY}"

## Step 5 UMI based deduplication using UMICollapse

This step removes PCR duplicates from aligned reads using unique molecular identifiers (UMIs).  
UMI based deduplication improves the accuracy of gene expression quantification by collapsing reads originating from the same original RNA molecule.

### Overview
- A SLURM array job is used to process samples in parallel
- Coordinate sorted BAM files from STAR alignment are used as input
- Deduplicated BAM files are generated and sorted by genomic coordinates

### Key parameters and options
- UMIs are parsed using an underscore separator
- A two pass strategy is applied to improve duplicate detection
- Deduplicated BAM files are sorted and indexed for downstream analyses

### Input and output
- **Input** Coordinate sorted BAM files from STAR alignment  
- **Output** UMI deduplicated, coordinate sorted, and indexed BAM files

> **Note**  
> File paths and SLURM resource specifications reflect a local computing environment and should be adapted as needed. The UMI separator should match the read naming convention used in the dataset.

In [ ]:
%%writefile /path/to/your/project/mapping/20260126_umicollapse_dedup_step.sb
#!/bin/bash
#SBATCH --array=0-5
#SBATCH --cpus-per-task=8
#SBATCH --time=12:00:00
#SBATCH --mem-per-cpu=3000mb
#SBATCH --output=/path/to/your/project/mapping/20260126_umicollapse_dedup_step.%a.out


QRYlist=(P55ST4_1_WT-1 P55ST4_2_WT-2 P55ST4_3_WT-3 P55ST4_4_S-1 P55ST4_5_S-2 P55ST4_6_S-3)
TASK_ID=${SLURM_ARRAY_TASK_ID:-}
if [[ -z "${TASK_ID}" ]]; then
  echo "ERROR. SLURM_ARRAY_TASK_ID is not set."
  exit 1
fi

QRY="${QRYlist[$TASK_ID]}"
if [[ -z "${QRY}" ]]; then
  echo "ERROR. No sample found for TASK_ID=${TASK_ID}"
  exit 1
fi

STAR_DIR=/path/to/your/project/mapping/star
DEDUP_DIR=/path/to/your/project/mapping/umicollapse
mkdir -p "$DEDUP_DIR"

IN_BAM="${STAR_DIR}/${QRY}.Aligned.sortedByCoord.out.bam"
if [[ ! -s "$IN_BAM" ]]; then
  echo "ERROR. Input BAM not found or empty. $IN_BAM"
  exit 1
fi

cd "$DEDUP_DIR" || exit 1

echo "Sample ${QRY}"
echo "Input  ${IN_BAM}"

RAW_DEDUP_BAM="${QRY}.umicollapse.dedup.bam"
FINAL_BAM="${QRY}.umicollapse.dedup.sortedByCoord.bam"
LOG="${QRY}.umicollapse.log"

umicollapse bam \
  -i "$IN_BAM" \
  -o "$RAW_DEDUP_BAM" \
  --umi-sep "_" \
  --two-pass \
  > "$LOG" 2>&1

if [[ ! -s "$RAW_DEDUP_BAM" ]]; then
  echo "ERROR. UMICollapse did not produce BAM for ${QRY}"
  echo "Check log ${LOG}"
  exit 1
fi

samtools sort -@ "${SLURM_CPUS_PER_TASK}" -o "$FINAL_BAM" "$RAW_DEDUP_BAM"
samtools index -@ "${SLURM_CPUS_PER_TASK}" "$FINAL_BAM"

rm -f "$RAW_DEDUP_BAM"

echo "Done. Output ${FINAL_BAM}"

## Step 6.1 Prepare gene model BED file for RSeQC

RSeQC requires a gene model annotation in 12-column BED format. The following commands convert the GTF annotation to BED format using UCSC utilities `gtfToGenePred` and `genePredToBed`.

> **Note**  
> File paths reflect a local computing environment. Adjust as needed for your system.

In [ ]:
%%bash
# Convert GTF annotation to 12-column BED format for RSeQC

GTF=/path/to/your/project/Genome_annotation/XL280_liftmerged_geneID_fixed.gtf
GENEPRED=/path/to/your/project/Genome_annotation/XL280.genePred
BED=/path/to/your/project/Genome_annotation/QC_gene_model.bed

gtfToGenePred \\
  "${GTF}" \\
  "${GENEPRED}"

genePredToBed \\
  "${GENEPRED}" \\
  "${BED}"


## Step 6.2 Alignment quality control using RSeQC, Qualimap, and MultiQC

This step performs post-alignment quality control on UMI-deduplicated RNA-seq data.  
Multiple complementary tools are applied to assess alignment quality, library characteristics, and annotation consistency across samples.

### Overview
- UMI-deduplicated, coordinate-sorted BAM files are used as input
- RSeQC is used to evaluate strand specificity, read distribution, and gene body coverage
- Qualimap provides additional RNA-seq QC metrics and visual reports
- MultiQC aggregates QC results across all samples into a single summary report

### Input and output
- **Input** UMI-deduplicated BAM files and a gene model BED file  
- **Output** Per-sample QC reports and a combined MultiQC summary report

> **Note**  
> The gene model BED file must be in 12-column format for compatibility with RSeQC.  
> File paths and SLURM resource specifications shown here reflect a local computing environment and may need to be adapted for other systems.

In [ ]:
%%writefile /path/to/your/project/qc/20260126_run_qc_rseqc_qualimap.sb
#!/bin/bash
#SBATCH --cpus-per-task=8
#SBATCH --time=12:00:00
#SBATCH --mem-per-cpu=6000mb
#SBATCH --output=/path/to/your/project/qc/20260126_run_qc_rseqc_qualimap.out

WORK=/path/to/your/project
BAM_DIR=${WORK}/mapping/umicollapse
QC_DIR=${WORK}/qc
RSEQC_DIR=${QC_DIR}/rseqc
QUALI_DIR=${QC_DIR}/qualimap
MULTIQC_DIR=${QC_DIR}/multiqc

mkdir -p "${RSEQC_DIR}" "${QUALI_DIR}" "${MULTIQC_DIR}"

SAMPLES=(P55ST4_1_WT-1 P55ST4_2_WT-2 P55ST4_3_WT-3 P55ST4_4_S-1 P55ST4_5_S-2 P55ST4_6_S-3)

# RSeQC needs a gene model BED in 12 column format
# Set this to your gene model bed file
GENE_BED=/path/to/your/project/Genome_annotation/QC_gene_model.bed

if [[ ! -s "${GENE_BED}" ]]; then
  echo "ERROR. RSeQC gene model BED not found. ${GENE_BED}"
  echo "You need a 12 column BED for RSeQC. If you tell me your GTF path I can give you the exact convert command."
  exit 1
fi

echo "Gene model bed: ${GENE_BED}"

for S in "${SAMPLES[@]}"; do
  BAM=${BAM_DIR}/${S}.umicollapse.dedup.sortedByCoord.bam
  if [[ ! -s "${BAM}" ]]; then
    echo "ERROR. Missing BAM for ${S}. ${BAM}"
    exit 1
  fi
  if [[ ! -s "${BAM}.bai" ]]; then
    echo "Index missing for ${S}. Building index."
    samtools index -@ "${SLURM_CPUS_PER_TASK}" "${BAM}"
  fi

  echo "Running RSeQC for ${S}"

  OUT_PREFIX=${RSEQC_DIR}/${S}

  # Strand specificity
  infer_experiment.py -i "${BAM}" -r "${GENE_BED}" > "${OUT_PREFIX}.infer_experiment.txt" 2>&1

  # Read distribution across features
  read_distribution.py -i "${BAM}" -r "${GENE_BED}" > "${OUT_PREFIX}.read_distribution.txt" 2>&1

  # Gene body coverage
  geneBody_coverage.py -i "${BAM}" -r "${GENE_BED}" -o "${OUT_PREFIX}.geneBody" > "${OUT_PREFIX}.geneBody_coverage.log" 2>&1

  echo "Running Qualimap for ${S}"

  SAMPLE_QDIR=${QUALI_DIR}/${S}
  mkdir -p "${SAMPLE_QDIR}"

  qualimap rnaseq \
    -bam "${BAM}" \
    -gtf /path/to/your/project/Genome_annotation/XL280_liftmerged_geneID_fixed.gtf \
    -outdir "${SAMPLE_QDIR}" \
    -outformat PDF:HTML \
    --java-mem-size=20G \
    > "${SAMPLE_QDIR}/${S}.qualimap.log" 2>&1

  echo "Finished ${S}"
done

echo "All per sample QC finished"

echo "Running MultiQC to combine everything"
multiqc "${QC_DIR}" -o "${MULTIQC_DIR}" -n 20260126_multiqc_report.html > "${MULTIQC_DIR}/multiqc.log" 2>&1

echo "Done. MultiQC report at ${MULTIQC_DIR}/20260126_multiqc_report.html"

## Step 7 Gene expression quantification using featureCounts

This step performs gene-level expression quantification using featureCounts on UMI-deduplicated, coordinate-sorted BAM files.

### Overview
- UMI-deduplicated BAM files are used as input
- Gene-level read counts are generated based on the provided GTF annotation
- Library layout (single-end or paired-end) is automatically inferred from the BAM files
- Strand specificity is evaluated by running featureCounts with both `-s 1` and `-s 2`, and the configuration with higher assigned read counts is selected

### Key features of this implementation
- Automatic detection of paired-end versus single-end data
- Conditional inclusion of `three_prime_UTR` features when present in the annotation
- Support for multi-mapping reads with fractional assignment
- Selection of optimal strand setting based on total assigned reads
* In this dataset, -s 1 consistently produced higher numbers of assigned reads and was therefore selected as the correct strand specific option.
  
### Input and output
- **Input** UMI-deduplicated, coordinate-sorted BAM files and a GTF annotation file  
- **Output** Gene-level count matrix, featureCounts summary files, and a record of the chosen strand configuration

> **Note**  
> The final gene-level count file is provided as a symbolic link to the selected featureCounts output.  
> File paths and resource specifications reflect a local computing environment and may need to be adapted for other systems.

In [ ]:
%%writefile /path/to/your/project/counts/20260126_featureCounts_full.sb
#!/bin/bash
#SBATCH --cpus-per-task=8
#SBATCH --time=06:00:00
#SBATCH --mem=32G
#SBATCH --output=/path/to/your/project/counts/20260126_featureCounts_full.%j.out


WORK=/path/to/your/project
BAM_DIR=${WORK}/mapping/umicollapse
OUT_DIR=${WORK}/counts
GTF=${WORK}/Genome_annotation/XL280_liftmerged_geneID_fixed.gtf

mkdir -p "${OUT_DIR}"

echo "Inputs"
echo "  GTF=${GTF}"
echo "  BAM_DIR=${BAM_DIR}"
echo "  OUT_DIR=${OUT_DIR}"

if [[ ! -s "${GTF}" ]]; then
  echo "ERROR GTF not found or empty ${GTF}"
  exit 1
fi

BAMS=$(ls ${BAM_DIR}/*.umicollapse.dedup.sortedByCoord.bam 2>/dev/null || true)
if [[ -z "${BAMS}" ]]; then
  echo "ERROR no BAM files found in ${BAM_DIR}"
  exit 1
fi

echo "BAM files"
ls -1 ${BAM_DIR}/*.umicollapse.dedup.sortedByCoord.bam

echo
echo "Detect SE or PE from first BAM"
FIRST_BAM=$(ls -1 ${BAM_DIR}/*.umicollapse.dedup.sortedByCoord.bam | head -n 1)
PAIR_FRAC=$(samtools view -@ ${SLURM_CPUS_PER_TASK} "${FIRST_BAM}" | head -n 20000 | awk '{if(and($2,1)) p++} END{print (p+0)/20000}')
echo "Paired flag fraction in first 20000 records ${PAIR_FRAC}"

PAIRED_OPT=""
awk -v x="${PAIR_FRAC}" 'BEGIN{exit (x>0.5)?0:1}' && PAIRED_OPT="-p" || true
if [[ "${PAIRED_OPT}" == "-p" ]]; then
  echo "Detected paired-end like BAM, will use -p"
else
  echo "Detected single-end like BAM, will not use -p"
fi

echo
echo "Check whether GTF contains three_prime_UTR feature"
FEATURES=$(cut -f3 "${GTF}" | sort | uniq | tr '\n' ' ')
echo "GTF feature types ${FEATURES}"

FT="-t exon"
if echo "${FEATURES}" | grep -qw "three_prime_UTR"; then
  FT="-t exon,three_prime_UTR"
  echo "Will use features exon and three_prime_UTR"
else
  echo "No three_prime_UTR found, will use exon only"
fi

echo
echo "Run featureCounts twice with -s 1 and -s 2, choose higher Assigned total"
OUT1=${OUT_DIR}/featureCounts_gene_level_s1.txt
OUT2=${OUT_DIR}/featureCounts_gene_level_s2.txt

featureCounts \
  -T ${SLURM_CPUS_PER_TASK} \
  ${PAIRED_OPT} \
  -s 1 \
  -M \
  --fraction \
  ${FT} \
  -g gene_id \
  -a "${GTF}" \
  -o "${OUT1}" \
  ${BAM_DIR}/*.umicollapse.dedup.sortedByCoord.bam

featureCounts \
  -T ${SLURM_CPUS_PER_TASK} \
  ${PAIRED_OPT} \
  -s 2 \
  -M \
  --fraction \
  ${FT} \
  -g gene_id \
  -a "${GTF}" \
  -o "${OUT2}" \
  ${BAM_DIR}/*.umicollapse.dedup.sortedByCoord.bam

SUM1=${OUT1}.summary
SUM2=${OUT2}.summary

if [[ ! -s "${SUM1}" || ! -s "${SUM2}" ]]; then
  echo "ERROR featureCounts summary missing"
  exit 1
fi

A1=$(grep -P "\tAssigned\t" "${SUM1}" | awk '{sum+=$2} END{print sum+0}')
A2=$(grep -P "\tAssigned\t" "${SUM2}" | awk '{sum+=$2} END{print sum+0}')

echo "Assigned total -s 1 ${A1}"
echo "Assigned total -s 2 ${A2}"

BEST=1
BEST_OUT=${OUT1}
BEST_SUM=${SUM1}
if [[ "${A2}" -gt "${A1}" ]]; then
  BEST=2
  BEST_OUT=${OUT2}
  BEST_SUM=${SUM2}
fi

FINAL_LINK=${OUT_DIR}/featureCounts_gene_level_FINAL.txt
FINAL_SUM_LINK=${OUT_DIR}/featureCounts_gene_level_FINAL.txt.summary

ln -sf "$(basename "${BEST_OUT}")" "${FINAL_LINK}"
ln -sf "$(basename "${BEST_SUM}")" "${FINAL_SUM_LINK}"

NOTE=${OUT_DIR}/featureCounts_strand_choice.txt
{
  echo "featureCounts Subread v2.1.1"
  echo "BAM input ${BAM_DIR}/*.umicollapse.dedup.sortedByCoord.bam"
  echo "GTF ${GTF}"
  echo "Feature types used ${FT}"
  echo "Paired option ${PAIRED_OPT:-none}"
  echo "Assigned -s 1 ${A1}"
  echo "Assigned -s 2 ${A2}"
  echo "Chosen -s ${BEST}"
  echo "Final counts ${FINAL_LINK}"
  echo "Final summary ${FINAL_SUM_LINK}"
} > "${NOTE}"

echo "Chosen -s ${BEST}"
echo "Final counts ${FINAL_LINK}"
echo "Final summary ${FINAL_SUM_LINK}"
echo "Note ${NOTE}"
echo "Done"